[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/15_cross_validation/90_exercises.ipynb)

# Bài tập — Chương 15: Cross-Validation (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/15_cross_validation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold, StratifiedKFold, TimeSeriesSplit, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Mô tả K-Fold, Stratified K-Fold, Group K-Fold, TimeSeriesSplit, LOO, nested CV — mỗi cái một câu, kèm khi nào dùng.

✍️ Trả lời:

**Đáp án:** K-Fold: chia $k$ phần, lần lượt mỗi phần làm validation — mẫu độc lập. Stratified: giữ tỉ lệ lớp trong mỗi fold — phân loại, nhất là mất cân bằng. Group K-Fold: mọi mẫu của một nhóm nằm cùng fold — nhiều mẫu mỗi bệnh nhân/khách hàng, dự đoán cho nhóm mới. TimeSeriesSplit: validation luôn sau train theo thời gian — dữ liệu thời gian/dự báo. LOO: $k = n$ — dữ liệu rất ít. Nested CV: vòng trong tune, vòng ngoài đánh giá cả quy trình — cần ước lượng không thiên lệch khi có tuning.

## L1 — Giải thích bằng lời

**Câu 2.** Một đồng nghiệp đạt ROC-AUC 0.97 bằng 5-fold CV trên dữ liệu ảnh X-quang (3 ảnh/bệnh nhân), sau khi đã chuẩn hoá ảnh và chọn 50 feature tốt nhất trên toàn bộ dữ liệu. Khi triển khai ở bệnh viện, ROC-AUC chỉ còn 0.71. Giải thích ít nhất 2 nguyên nhân từ quy trình CV.

✍️ Trả lời:

**Gợi ý:** (1) **Rò rỉ nhóm**: KFold ngẫu nhiên đặt ảnh của cùng bệnh nhân vào cả train và validation → model nhận ra bệnh nhân thay vì bệnh → cần GroupKFold theo bệnh nhân. (2) **Chọn feature trên toàn bộ dữ liệu** trước CV → validation đã tham gia chọn feature → lạc quan; phải để trong Pipeline. (3) Chuẩn hoá ảnh theo thống kê toàn bộ dữ liệu cũng là rò rỉ nhẹ. (4) Ngoài CV: khác biệt phân phối giữa bệnh viện (máy chụp, quần thể bệnh nhân) — cần validation theo bệnh viện (group = bệnh viện).

## L2 — Tính tay

**Câu 3.** Dữ liệu 103 mẫu, 5-fold CV (chia như `np.array_split`): kích thước các fold validation? Mỗi model train trên bao nhiêu mẫu? Với TimeSeriesSplit `n_splits=4`, `test_size=20` trên 103 mẫu, split cuối train trên bao nhiêu mẫu và validation là các chỉ số nào?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
# STARTER: fold_sizes = None      # list 5 kích thước fold validation
# STARTER: train_sizes = None     # list 5 kích thước train tương ứng
# STARTER: last_train_size, last_val = None, None   # số mẫu train của split cuối, (chỉ số val đầu, chỉ số val cuối)
fold_sizes = [len(f) for f in np.array_split(np.arange(103), 5)]
train_sizes = [103 - s for s in fold_sizes]
last_train_size, last_val = 103 - 20, (83, 102)
### END CODE HERE ###

In [ ]:
check_equal(fold_sizes, [21, 21, 21, 20, 20], "kích thước fold")
check_equal(train_sizes, [82, 82, 82, 83, 83], "kích thước train")
tr, va = list(TimeSeriesSplit(4, test_size=20).split(np.zeros(103)))[-1]
check_equal(last_train_size, len(tr), "split cuối: số mẫu train")
check_equal(last_val, (va.min(), va.max()), "split cuối: chỉ số validation")

**Câu 4.** 5 fold cho accuracy `[0.82, 0.85, 0.79, 0.88, 0.81]`. Tính trung bình, độ lệch chuẩn (ddof = 1) và sai số chuẩn xấp xỉ $s/\sqrt k$.

In [ ]:
scores4 = np.array([0.82, 0.85, 0.79, 0.88, 0.81])
### START CODE HERE ### (≈ 3 dòng)
mean4 = scores4.mean()
std4 = scores4.std(ddof=1)
se4 = std4 / np.sqrt(len(scores4))
### END CODE HERE ###

In [ ]:
check_close(mean4, 0.83, message="trung bình")
check_close(std4, np.sqrt(0.005 / 4), tol=1e-6, message="độ lệch chuẩn (ddof=1): tổng bình phương độ lệch = 0.005")
check_close(se4, std4 / np.sqrt(5), message="sai số chuẩn")

## L3 — Code

**Bài 5.** Viết `stratified_kfold(y, k, seed)` trả về danh sách `(train_idx, val_idx)`: với **mỗi lớp**, xáo trộn chỉ số rồi chia đều thành $k$ phần; fold $i$ = hợp các phần thứ $i$ của mọi lớp. Kiểm tra mỗi fold giữ tỉ lệ lớp.

In [ ]:
def stratified_kfold(y, k=5, seed=0):
    r = np.random.default_rng(seed)
    folds = [[] for _ in range(k)]
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: # với mỗi lớp: lấy chỉ số, r.shuffle, np.array_split thành k phần, phần i thêm vào folds[i]
    for label in np.unique(y):
        idx = np.flatnonzero(y == label)
        r.shuffle(idx)
        for i, chunk in enumerate(np.array_split(idx, k)):
            folds[i].extend(chunk.tolist())
    ### END CODE HERE ###
    all_idx = np.arange(len(y))
    return [(np.setdiff1d(all_idx, f), np.sort(np.array(f))) for f in folds]

In [ ]:
y5 = (rng.random(1000) < 0.1).astype(int)
splits = stratified_kfold(y5, k=5)
rates = [y5[val].mean() for _, val in splits]
check_less(max(rates) - min(rates), 0.01, "tỉ lệ positive mỗi fold gần như bằng nhau")
check_equal(sorted(np.concatenate([val for _, val in splits]).tolist()), list(range(1000)), "mỗi mẫu xuất hiện đúng một lần trong validation")
check_true(all(len(np.intersect1d(tr, va)) == 0 for tr, va in splits), "train và validation không giao nhau")

**Bài 6.** Viết `group_kfold(groups, k)` — gán mỗi nhóm vào fold đang có **ít mẫu nhất** (xét nhóm theo thứ tự kích thước giảm dần), trả về danh sách `(train_idx, val_idx)`.

In [ ]:
def group_kfold(groups, k=5):
    uniq, counts = np.unique(groups, return_counts=True)
    load = np.zeros(k)
    fold_of = {}
    ### START CODE HERE ### (≈ 4 dòng)
    # STARTER: # duyệt nhóm theo kích thước giảm dần; gán vào fold có load nhỏ nhất; cập nhật load
    for g in uniq[np.argsort(-counts, kind="stable")]:
        f = int(np.argmin(load))
        fold_of[g] = f
        load[f] += counts[uniq == g][0]
    ### END CODE HERE ###
    fold_ids = np.array([fold_of[g] for g in groups])
    return [(np.flatnonzero(fold_ids != f), np.flatnonzero(fold_ids == f)) for f in range(k)]

In [ ]:
groups6 = rng.integers(0, 40, 800)
splits6 = group_kfold(groups6, k=5)
check_true(all(len(np.intersect1d(groups6[tr], groups6[va])) == 0 for tr, va in splits6), "không nhóm nào nằm ở cả train và val")
sizes6 = [len(va) for _, va in splits6]
check_less(max(sizes6) - min(sizes6), 40, "các fold có kích thước gần nhau")

## L4 — Debug

**Bài 7.** Đoạn code sau cho ROC-AUC "đẹp" trên dữ liệu có **nhãn ngẫu nhiên**. Tìm và sửa **2 lỗi**: (1) `StandardScaler` + chọn feature (`SelectKBest`) fit trên toàn bộ dữ liệu trước CV; (2) báo cáo `best_score_` của GridSearchCV như hiệu năng. Sửa: Pipeline (scaler + SelectKBest + model) và **nested CV** — lưu kết quả vào `honest_auc`.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X7 = rng.normal(size=(120, 2000))
y7 = rng.integers(0, 2, 120)
### START CODE HERE ### (≈ 6 dòng)
# STARTER: X7_s = StandardScaler().fit_transform(X7)                                  # 🐛 (1)
# STARTER: X7_sel = SelectKBest(f_classif, k=10).fit_transform(X7_s, y7)               # 🐛 (1)
# STARTER: grid = GridSearchCV(LogisticRegression(max_iter=2000), {"C": [0.01, 0.1, 1, 10]}, cv=5, scoring="roc_auc").fit(X7_sel, y7)
# STARTER: honest_auc = grid.best_score_                                               # 🐛 (2)
pipe = make_pipeline(StandardScaler(), SelectKBest(f_classif, k=10), LogisticRegression(max_iter=2000))
grid = GridSearchCV(pipe, {"logisticregression__C": [0.01, 0.1, 1, 10]}, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                    scoring="roc_auc")
honest_auc = cross_val_score(grid, X7, y7, cv=StratifiedKFold(5, shuffle=True, random_state=1), scoring="roc_auc").mean()
### END CODE HERE ###
print(f"ROC-AUC trung thực = {honest_auc:.3f}")

In [ ]:
check_less(abs(honest_auc - 0.5), 0.1, "nhãn ngẫu nhiên → ROC-AUC ≈ 0.5")

## L5 — Phân tích

**Bài 8.** Đo **variance của ước lượng CV** theo $k$: với dữ liệu churn (`customer_churn.csv`, 600 mẫu đầu), Logistic Regression, tính ROC-AUC trung bình của $k$-fold CV với 30 cách xáo trộn khác nhau, $k \in \{2, 5, 10\}$. Lưu DataFrame `res8` (cột `k`, `mean`, `std_across_shuffles`). Nhận xét bias (so với ROC-AUC trên phần dữ liệu còn lại — `reference_auc`) và variance.

In [ ]:
df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
Xc = pd.get_dummies(df.drop(columns=["customer_id", "churn"]), dtype=float).to_numpy()
yc = df["churn"].to_numpy()
X8, y8, X_ref, y_ref = Xc[:600], yc[:600], Xc[600:], yc[600:]
model8 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
reference_auc = roc_auc_score(y_ref, model8.fit(X8, y8).predict_proba(X_ref)[:, 1])
rows8 = []
### START CODE HERE ### (≈ 5 dòng)
# STARTER: # với mỗi k: 30 giá trị ROC-AUC trung bình của StratifiedKFold(k, shuffle=True, random_state=s), s = 0..29
# STARTER: # rows8.append({"k": k, "mean": ..., "std_across_shuffles": ...})
for k in [2, 5, 10]:
    means = [cross_val_score(model8, X8, y8, cv=StratifiedKFold(k, shuffle=True, random_state=s), scoring="roc_auc").mean()
             for s in range(30)]
    rows8.append({"k": k, "mean": np.mean(means), "std_across_shuffles": np.std(means)})
### END CODE HERE ###
res8 = pd.DataFrame(rows8)
print(f"ROC-AUC tham chiếu (model train trên 600 mẫu, đánh giá trên {len(y_ref)} mẫu còn lại) = {reference_auc:.4f}")
res8.round(4)

In [ ]:
check_greater(res8.loc[res8.k == 2, "std_across_shuffles"].item(), res8.loc[res8.k == 10, "std_across_shuffles"].item(),
              "k = 2 dao động theo cách xáo trộn nhiều hơn k = 10")
check_less(res8.loc[res8.k == 2, "mean"].item(), res8.loc[res8.k == 10, "mean"].item() + 0.002, "k = 2 không lạc quan hơn k = 10")

✍️ Nhận xét:

**Gợi ý:** $k = 2$ train trên 50% dữ liệu → ước lượng hơi **bi quan** (model yếu hơn model cuối train trên 100%), và dao động theo cách chia nhiều hơn (mỗi lần chỉ trung bình 2 điểm số). $k = 5$–10 gần với giá trị tham chiếu và ổn định hơn. Tăng $k$ lớn hơn nữa tốn thời gian mà lợi ích giảm dần.

## L6 — Thực tế

**Bài 9.** Dự báo doanh số ngày (`sales_daily.csv`): tạo feature lag (`lag_1`, `lag_7`), trung bình trượt 7 ngày **đã dịch 1 ngày** (`roll7`), thứ trong tuần, khuyến mãi. Chọn `alpha` của Ridge bằng `TimeSeriesSplit(5, gap=0)` trên 80% đầu; đánh giá MAE trên 20% cuối (`future_mae`). So với baseline "doanh số tuần trước cùng thứ" (`naive_mae` = dùng `lag_7`). Yêu cầu: model tốt hơn baseline.

In [ ]:
sales = pd.read_csv(repo_path("data", "sample", "sales_daily.csv"), parse_dates=["date"])
print(sales.head(3))
### START CODE HERE ### (≈ 12 dòng)
# STARTER: d = None            # sales + cột lag_1, lag_7, roll7 (shift(1).rolling(7).mean()), dow one-hot; bỏ hàng NaN
# STARTER: feats = None        # các cột feature (trừ date, sales)
# STARTER: cut = None          # 80% đầu
# STARTER: X_hist, y_hist, X_fut, y_fut = None, None, None, None
# STARTER: search9 = None      # GridSearchCV(Pipeline(StandardScaler, Ridge), alpha logspace, cv=TimeSeriesSplit(5), MAE)
# STARTER: future_mae = None
# STARTER: naive_mae = None    # MAE khi dự đoán bằng lag_7
d = sales.copy()
d["lag_1"] = d["sales"].shift(1)
d["lag_7"] = d["sales"].shift(7)
d["roll7"] = d["sales"].shift(1).rolling(7).mean()
d["dow"] = d["date"].dt.dayofweek
d = pd.get_dummies(d.dropna(), columns=["dow"], dtype=float)
feats = [c for c in d.columns if c not in ("date", "sales")]
cut = int(0.8 * len(d))
X_hist, y_hist, X_fut, y_fut = d[feats].iloc[:cut], d["sales"].iloc[:cut], d[feats].iloc[cut:], d["sales"].iloc[cut:]
search9 = GridSearchCV(make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": np.logspace(-2, 3, 11)},
                       cv=TimeSeriesSplit(5), scoring="neg_mean_absolute_error").fit(X_hist, y_hist)
future_mae = mean_absolute_error(y_fut, search9.predict(X_fut))
naive_mae = mean_absolute_error(y_fut, X_fut["lag_7"])
### END CODE HERE ###
print(f"alpha = {search9.best_params_['ridge__alpha']:.3g} | MAE tương lai: Ridge = {future_mae:.2f} | baseline lag-7 = {naive_mae:.2f}")

In [ ]:
check_less(future_mae, naive_mae, "model tốt hơn baseline 'tuần trước cùng thứ'")

## L7 — Nâng cao

**Bài 10.** Viết `purged_time_series_split(n, n_splits, test_size, gap)` — expanding window với `gap` mẫu bị **loại bỏ** giữa train và validation (ý tưởng *purging* của López de Prado). Khi target **nhìn về tương lai** (vd target của ngày $t$ = tổng doanh số các ngày $t+1, \dots, t+7$), các mẫu train cuối cùng có target **chồng lấn thời gian** với validation → rò rỉ. Viết thêm `count_label_overlap(train_idx, val_idx, horizon)` đếm số mẫu train có cửa sổ target $[t+1, t+\text{horizon}]$ giao với khoảng thời gian của validation. Chứng minh: `gap = 0` → mỗi split có 7 mẫu rò rỉ; `gap = 7` → 0.

In [ ]:
def purged_time_series_split(n, n_splits=5, test_size=60, gap=0):
    ### START CODE HERE ### (≈ 5 dòng)
    out = []
    for i in range(n_splits):
        val_start = n - (n_splits - i) * test_size
        out.append((np.arange(0, val_start - gap), np.arange(val_start, val_start + test_size)))
    return out
    ### END CODE HERE ###


def count_label_overlap(train_idx, val_idx, horizon=7):
    ### START CODE HERE ### (≈ 2 dòng)
    # STARTER: label_end = None     # ngày cuối trong cửa sổ target của mỗi mẫu train
    # STARTER: return None          # số mẫu có [t+1, label_end] giao [val.min(), val.max()]
    label_end = train_idx + horizon                       # cửa sổ target của mẫu t: [t+1, t+horizon]
    return int(np.sum((label_end >= val_idx.min()) & (train_idx + 1 <= val_idx.max())))
    ### END CODE HERE ###

In [ ]:
check_equal([(tr.max(), va.min()) for tr, va in purged_time_series_split(200, 3, 40, gap=5)], [(74, 80), (114, 120), (154, 160)],
            "train kết thúc trước validation đúng gap mẫu")
overlap = {gap: [count_label_overlap(tr, va, horizon=7) for tr, va in purged_time_series_split(1000, 5, 60, gap=gap)] for gap in [0, 3, 7]}
print("số mẫu train có target chồng lấn validation, theo gap:", overlap)
check_equal(overlap[0], [7] * 5, "gap = 0: 7 mẫu rò rỉ mỗi split")
check_equal(overlap[7], [0] * 5, "gap = horizon = 7: không còn rò rỉ")

✍️ Vì sao `gap` cần ít nhất bằng **horizon** của target (và của các feature nhìn về tương lai nếu có)?

**Gợi ý:** target của mẫu $t$ được tính từ dữ liệu tới ngày $t + h$. Nếu train kết thúc ở ngày $v - 1$ ngay trước validation bắt đầu ở $v$, thì target của các mẫu $t \in [v - h, v - 1]$ chứa doanh số của những ngày nằm **trong** validation → model đã "nhìn thấy" một phần đáp án của validation. Bỏ đi $h$ mẫu cuối (gap = h) loại hoàn toàn chồng lấn. Tương tự với feature rolling không dịch hoặc dữ liệu có độ trễ công bố.